In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

In [30]:
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

In [31]:
device = ("cuda" if torch.cuda.is_available() else "cpu")
device

'cuda'

In [32]:
normal_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.255])
])
verticle_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop((224,224)),
    transforms.RandomVerticalFlip(p=1.0),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.255])
])
horizontal_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop((224,224)),
    transforms.RandomVerticalFlip(p=1.0),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.255])
])

In [33]:
import glob
import os

image_folder_path = 'Cat_And_Dog/train'

all_image_paths = glob.glob(os.path.join(image_folder_path,"*jpg"))

X_paths = []
y_labels = []

for path in all_image_paths:
    filename = os.path.basename(path).lower()
    if "cat" in filename:
        X_paths.append(path)
        y_labels.append(0)
    elif "dog" in filename:
        X_paths.append(path)
        y_labels.append(1)

In [34]:
len(X_paths)

19778

In [35]:
len(y_labels)

19778

In [36]:
X_train_path,X_test_path,y_train_path,y_test_path = train_test_split(X_paths,y_labels,random_state=42,test_size=0.2,stratify=y_labels)

In [37]:
from PIL import Image
class customDataset (Dataset):
    def __init__(self,file_paths,labels,transform):
        self.labels = labels
        self.file_paths = file_paths
        self.transform = transform
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, index):
        image = Image.open(self.file_paths[index]).convert("RGB")
        image = self.transform(image)
        return image , torch.tensor(self.labels[index],dtype=torch.long)

In [38]:
train_base = customDataset(X_train_path,y_train_path,normal_transform)
train_h_base = customDataset(X_train_path,y_train_path,horizontal_transform)
train_v_base = customDataset(X_train_path,y_train_path,verticle_transform)

In [39]:
from torch.utils.data import ConcatDataset
tripled_train_dataset = ConcatDataset([train_base,train_h_base,train_v_base])
test_dataset = customDataset(X_test_path,y_test_path,normal_transform)

In [40]:
train_loader = DataLoader(tripled_train_dataset,batch_size=32,shuffle=True,pin_memory=True,num_workers=6)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True,num_workers=6)

In [41]:
from torchvision import models
weights = models.ResNet50_Weights.DEFAULT
resnet = models.resnet50(weights=weights)

In [42]:
for param in resnet.parameters():
    param.requires_grad = False

In [43]:
for param in resnet.layer4.parameters():
    param.requires_grad = True

In [44]:
resnet.fc = nn.Sequential(
    nn.Linear(2048,512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512,2)
)

In [45]:
resnet.to(device=device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [46]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(filter(lambda p: p.requires_grad,resnet.parameters()),lr=2e-4,weight_decay=1e-3)

In [47]:
epochs = 10
schedular = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [48]:
from tqdm import tqdm
for epoch in range(epochs):
    resnet.train()
    total_epochs_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}", unit="batch")

    for batch_features , batch_labels in progress_bar:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        outputs = resnet(batch_features)
        loss = criterion(outputs,batch_labels)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(resnet.parameters(),max_norm=1.0)
        optimizer.step()
        total_epochs_loss +=loss.item()

        progress_bar.set_postfix(batch_loss=f"{loss.item():.4f}")
    
    schedular.step()
    avg_loss = total_epochs_loss/len(train_loader)
    print(f'Epoch: {epoch + 1} Done! Average Training Loss: {avg_loss:.4f}\n')

Epoch 1/10:   1%|          | 17/1484 [00:02<02:13, 10.99batch/s, batch_loss=0.1278]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 1/10:  29%|██▉       | 431/1484 [00:37<01:30, 11.66batch/s, batch_loss=0.0143]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 1/10:  54%|█████▍    | 807/1484 [01:09<00:58, 11.63batch/s, batch_loss=0.0282]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 1/10: 100%|██████████| 1484/1484 [02:08<00:00, 11.59batch/s, batch_loss=0.3733]


Epoch: 1 Done! Average Training Loss: 0.0667



Epoch 2/10:  65%|██████▌   | 967/1484 [01:23<00:44, 11.63batch/s, batch_loss=0.0003]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 2/10: 100%|██████████| 1484/1484 [02:08<00:00, 11.59batch/s, batch_loss=0.0001]


Epoch: 2 Done! Average Training Loss: 0.0223



Epoch 3/10:   7%|▋         | 101/1484 [00:09<01:59, 11.57batch/s, batch_loss=0.0109]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 3/10:  19%|█▉        | 289/1484 [00:25<01:43, 11.55batch/s, batch_loss=0.0095]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 3/10:  88%|████████▊ | 1307/1484 [01:53<00:15, 11.57batch/s, batch_loss=0.0024]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 3/10: 100%|██████████| 1484/1484 [02:08<00:00, 11.55batch/s, batch_loss=0.0001]


...

Epoch: 8 Done! Average Training Loss: 0.0008



Epoch 9/10:   2%|▏         | 27/1484 [00:03<02:09, 11.28batch/s, batch_loss=0.0000]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 9/10:  81%|████████  | 1195/1484 [01:44<00:25, 11.34batch/s, batch_loss=0.0000]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 9/10:  81%|████████  | 1201/1484 [01:45<00:24, 11.45batch/s, batch_loss=0.0002]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 9/10: 100%|██████████| 1484/1484 [02:09<00:00, 11.43batch/s, batch_loss=0.0007]


Epoch: 9 Done! Average Training Loss: 0.0005



Epoch 10/10:  12%|█▏        | 185/1484 [00:16<01:53, 11.43batch/s, batch_loss=0.0000]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 10/10:  89%|████████▉ | 1327/1484 [01:56<00:13, 11.40batch/s, batch_loss=0.0000]/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Epoch 10/10: 100%|██████████| 1484/1484 [02:10<00:00, 11.41batch/s, batch_loss=0.0000]

Epoch: 10 Done! Average Training Loss: 0.0006



In [49]:
resnet.eval()
total = 0
correct = 0
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        outputs = resnet(batch_features)
        _,predicted = torch.max(outputs,1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()
accuracy = correct / total
print(f"Final Flat-Folder Cats vs Dogs Test Accuracy: {accuracy * 100:.2f}%")

Final Flat-Folder Cats vs Dogs Test Accuracy: 99.12%


In [50]:
image_folder_path = 'Cat_And_Dog/test'

all_image_paths = glob.glob(os.path.join(image_folder_path,"*jpg"))

X_paths_test = []
y_labels_test = []

for path in all_image_paths:
    filename = os.path.basename(path).lower()
    if "cat" in filename:
        X_paths_test.append(path)
        y_labels_test.append(0)
    elif "dog" in filename:
        X_paths_test.append(path)
        y_labels_test.append(1)

In [51]:
len(X_paths_test)

5181

In [52]:
test_new_dataset = customDataset(X_paths_test,y_labels_test,normal_transform)

In [53]:
test_new_dataloader = DataLoader(test_new_dataset,batch_size=32,pin_memory=True,shuffle=False,num_workers=4)

In [54]:
resnet.eval()
total = 0
correct = 0
with torch.no_grad():
    for batch_features, batch_labels in test_new_dataloader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        outputs = resnet(batch_features)
        _,predicted = torch.max(outputs,1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()
accuracy = correct / total
print(f"Final Flat-Folder Cats vs Dogs Test Accuracy: {accuracy * 100:.2f}%")

Final Flat-Folder Cats vs Dogs Test Accuracy: 99.05%


In [55]:
Model_Save_Path = "resnet_cats_dogs.pth"
torch.save(resnet.state_dict(),Model_Save_Path)
print(f"Model weights successfully saved to {Model_Save_Path}")

Model weights successfully saved to resnet_cats_dogs.pth


In [56]:
resnet_eval = models.resnet50()
resnet_eval.fc = nn.Sequential(
    nn.Linear(2048, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 2)
)

saved_weights = torch.load("resnet_cats_dogs.pth", map_location=device)

resnet_eval.load_state_dict(saved_weights)

resnet_eval.to(device)
resnet_eval.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [57]:
image_folder_path = 'Cat_And_Dog/test1/cat'

all_image_paths = glob.glob(os.path.join(image_folder_path,"*jpg"))

X_paths_test = []
y_labels_test = []

for path in all_image_paths:
    filename = os.path.basename(path).lower()
    if "cat" in filename:
        X_paths_test.append(path)
        y_labels_test.append(0)
    elif "dog" in filename:
        X_paths_test.append(path)
        y_labels_test.append(1)
image_folder_path = 'Cat_And_Dog/test1/dog'

all_image_paths = glob.glob(os.path.join(image_folder_path,"*jpg"))

for path in all_image_paths:
    filename = os.path.basename(path).lower()
    if "cat" in filename:
        X_paths_test.append(path)
        y_labels_test.append(0)
    elif "dog" in filename:
        X_paths_test.append(path)
        y_labels_test.append(1)

In [58]:
test_new_dataset = customDataset(X_paths_test,y_labels_test,normal_transform)

In [59]:
test_new_dataloader = DataLoader(test_new_dataset,batch_size=32,pin_memory=True,shuffle=False,num_workers=4)

In [60]:
resnet.eval()
total = 0
correct = 0
with torch.no_grad():
    for batch_features, batch_labels in test_new_dataloader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        outputs = resnet_eval(batch_features)
        _,predicted = torch.max(outputs,1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()
accuracy = correct / total
print(f"Final Flat-Folder Cats vs Dogs Test Accuracy: {accuracy * 100:.2f}%")

Final Flat-Folder Cats vs Dogs Test Accuracy: 99.05%


In [61]:
resnet.eval()
total = 0
correct = 0
with torch.no_grad():
    for batch_features, batch_labels in train_loader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        outputs = resnet_eval(batch_features)
        _,predicted = torch.max(outputs,1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()
accuracy = correct / total
print(f"Final Flat-Folder Cats vs Dogs Test Accuracy: {accuracy * 100:.2f}%")

/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/home/ujwal/miniconda3/envs/pytorch_env/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Final Flat-Folder Cats vs Dogs Test Accuracy: 99.98%
